# Pandas – Teil 1: DataFrames verstehen und Daten auswählen

`pandas` ist eine der wichtigsten Python-Bibliotheken für **tabellarische Daten**.

NumPy-Arrays sind besonders gut geeignet, wenn viele gleichartige numerische Werte effizient verarbeitet werden sollen. Ein `DataFrame` ergänzt dazu unter anderem:

- benannte Spalten,
- einen Index für die Zeilen,
- unterschiedliche Datentypen in verschiedenen Spalten,
- viele Werkzeuge zum Filtern, Sortieren und Zusammenfassen von Daten.

Üblicherweise importieren wir pandas so:

In [ ]:
import pandas as pd

## Kurz zum Notebook

In VS Code kann eine `.ipynb`-Datei Zelle für Zelle ausgeführt werden. Wählt dafür den Python-Interpreter beziehungsweise Kernel eures Projekts, normalerweise aus `.venv`.

Die Zellen teilen sich einen gemeinsamen Zustand: Variablen aus einer bereits ausgeführten Zelle können in späteren Zellen weiterverwendet werden. Für einen reproduzierbaren Ablauf ist es deshalb sinnvoll, das Notebook **von oben nach unten** auszuführen.

## Ein erstes `DataFrame`

Wir beginnen mit einer kleinen, bewusst einfachen Tabelle. Ein Dictionary eignet sich gut, um ein `DataFrame` zu erzeugen: Die Schlüssel werden zu Spaltennamen.

In [ ]:
cities = pd.DataFrame(
    {
        "city": ["Essen", "Bochum", "Dortmund"],
        "inhabitants": [591032, 364454, 587696],
        "area_km2": [210.3, 145.7, 280.7],
    },
    index=["E", "B", "D"],
)

cities

Ein `DataFrame` ist zweidimensional: Es hat Zeilen und Spalten. Zusätzlich besitzt es Beschriftungen:

- **columns**: Namen der Spalten
- **index**: Beschriftungen der Zeilen

Anders als bei einem NumPy-Array müssen nicht alle Spalten denselben Datentyp haben.

In [ ]:
print(cities.index)
print(cities.columns)
print(cities.dtypes)
print(cities.shape)

## `Series` und `DataFrame`

Eine einzelne Spalte wird in pandas meistens als **`Series`** zurückgegeben. Eine `Series` ist eine eindimensionale Datenstruktur mit Index.

Der Unterschied zwischen diesen beiden Ausdrücken ist wichtig:

In [ ]:
one_column = cities["city"]
one_column_as_table = cities[["city"]]

print(type(one_column))
print(type(one_column_as_table))

one_column

In [ ]:
one_column_as_table

`cities["city"]` liefert eine `Series`.

`cities[["city"]]` bekommt dagegen eine **Liste von Spaltennamen** und liefert deshalb wieder ein `DataFrame` – hier eben mit nur einer Spalte.

## Zeilen und Spalten auswählen: `loc` und `iloc`

Mit `.loc[...]` wählen wir anhand von **Labels** aus.

Mit `.iloc[...]` wählen wir anhand von **Positionen** aus.

In [ ]:
cities.loc["B"]

In [ ]:
cities.loc[["E", "D"], ["city", "inhabitants"]]

Eine Besonderheit von `.loc`: Bei einem Slice mit Labels ist die **Stopgrenze enthalten**.

In [ ]:
cities.loc["E":"D", "city":"inhabitants"]

`.iloc` verhält sich dagegen wie die bekannten Python-Slices: Die Stopposition ist **nicht enthalten**.

In [ ]:
cities.iloc[0:2, 0:2]

Als Kurzfassung:

- `df["spalte"]` → eine Spalte als `Series`
- `df[["spalte1", "spalte2"]]` → mehrere Spalten als `DataFrame`
- `df.loc[zeilen_labels, spalten_labels]` → Auswahl über Labels
- `df.iloc[zeilen_positionen, spalten_positionen]` → Auswahl über Positionen

## Spalten hinzufügen und entfernen

Neue Spalten können direkt berechnet werden. Die Rechnung wird elementweise auf die Spalte angewendet.

In [ ]:
cities["inhabitants_millions"] = cities["inhabitants"] / 1_000_000

cities

Mit `.drop(columns=...)` können wir eine neue Tabelle ohne bestimmte Spalten erzeugen. Ohne weitere Optionen bleibt das ursprüngliche `DataFrame` dabei erhalten.

In [ ]:
cities_small = cities.drop(columns="area_km2")

cities_small

## Eine CSV-Datei einlesen

Jetzt wechseln wir zu einem größeren Datensatz. Dabei verwenden wir bewusst wieder `os.path.join()`, wie im vorherigen Kapitel zu Ordnern und Pfaden.

Die Datei `pokemon.csv` liegt im Kursprojekt im Ordner `data`.

In [ ]:
import os

data_path = os.path.join("..", "data", "pokemon.csv")

pokemon = pd.read_csv(data_path, encoding="utf-8")

Nach dem Einlesen sollten wir nicht sofort komplizierte Analysen starten. Zuerst verschaffen wir uns einen Überblick.

In [ ]:
pokemon.head()

In [ ]:
pokemon.tail(3)

In [ ]:
print("Form:", pokemon.shape)
print("Spalten:")
print(pokemon.columns)

`.dtypes` zeigt den Datentyp jeder Spalte. `info()` fasst zusätzlich unter anderem die Anzahl nicht-fehlender Einträge zusammen.

In [ ]:
pokemon.dtypes

In [ ]:
pokemon.info()

## Ein kleineres Arbeits-DataFrame

Der Datensatz besitzt viele Spalten. Für die nächsten Schritte wählen wir nur einige davon aus.

In [ ]:
columns = [
    "name",
    "type1",
    "type2",
    "speed",
    "attack",
    "defense",
    "hp",
    "weight_kg",
    "generation",
    "is_legendary",
]

pokemon_small = pokemon[columns]

pokemon_small.head()

## Erste Kennzahlen

`.describe()` fasst numerische Spalten zusammen. Unter anderem sehen wir Anzahl, Mittelwert, Standardabweichung, Minimum, Quartile und Maximum.

In [ ]:
pokemon_small.describe()

Bei kategorialen Spalten sind häufig andere Fragen interessant:

- Welche verschiedenen Werte gibt es?
- Wie oft kommt jeder Wert vor?

In [ ]:
pokemon_small["type1"].unique()

In [ ]:
pokemon_small["type1"].value_counts()

## Mit Bedingungen auswählen

Die booleschen Masken aus NumPy funktionieren auch sehr gut mit pandas.

Zuerst erzeugen wir nur die Bedingung:

In [ ]:
fast_mask = pokemon_small["speed"] >= 100

fast_mask.head()

Die Maske kann anschließend mit `.loc` benutzt werden. So können wir gleichzeitig passende Zeilen **und** bestimmte Spalten auswählen.

In [ ]:
pokemon_small.loc[
    fast_mask,
    ["name", "type1", "speed"],
].head(10)

Mehrere Bedingungen werden – genau wie bei NumPy – mit `&` und `|` kombiniert. Jede Teilbedingung wird geklammert.

In [ ]:
strong_and_fast = (
    (pokemon_small["speed"] >= 100)
    & (pokemon_small["attack"] >= 100)
)

pokemon_small.loc[
    strong_and_fast,
    ["name", "type1", "speed", "attack"],
].head(10)

## Sortieren

`.sort_values()` sortiert nach den Werten einer oder mehrerer Spalten.

In [ ]:
pokemon_small.sort_values(
    "speed",
    ascending=False,
).head(10)

Bei mehreren Sortierschlüsseln können wir für jede Spalte eine eigene Richtung angeben. Hier wird zuerst nach `speed` absteigend und bei gleichem Tempo nach `name` aufsteigend sortiert.

In [ ]:
pokemon_small.sort_values(
    ["speed", "name"],
    ascending=[False, True],
).head(10)

## Falls wirklich ein NumPy-Array benötigt wird

Meistens ist es sinnvoll, beim Arbeiten mit tabellarischen Daten im `DataFrame` beziehungsweise in der `Series` zu bleiben.

Falls eine andere Bibliothek ausdrücklich ein NumPy-Array erwartet, verwenden wir `.to_numpy()`:

In [ ]:
speed_array = pokemon_small["speed"].to_numpy()

print(type(speed_array))
print(speed_array[:10])

Dabei gehen pandas-spezifische Informationen wie der Spaltenname und der Index verloren.

## Zwischenfazit

Die wichtigsten Muster aus diesem ersten Teil sind:

```python
df["column"]
df[["column1", "column2"]]

df.loc[rows, columns]
df.iloc[row_positions, column_positions]

df.head()
df.shape
df.dtypes
df.info()

df["column"].value_counts()
df.loc[mask, columns]
df.sort_values(...)
```

Im zweiten Teil schauen wir uns fehlende Werte, gezieltes Verändern und das Zusammenfassen von Gruppen an.